## Case study: étude de l'offre et du positionnement de Freeglisse

**_Contexte_**

Olist aimerait désormais **augmenter son offre sur le segment du sport**, en incitant des marketplaces d'articles de sport à rejoindre la plateforme. Pour affiner sa stratégie vis-à-vis de ce type de vendeurs, Olist a besoin **d'analyser leur offre et leur positionnement**, pour mieux connaitre le marché sur lequel ils évoluent.

En particulier, les Sales d'Olist s'intéressent à **FreeGlisse**, une plateforme de vente en ligne de matériel de ski d'occasion. Leur ojectif est d'emmagasiner le plus d’informations possibles à partir de ce que FreeGlisse affiche sur son site web : https://freeglisse.com/fr/12-ski-occasion
<br><br>

**_Instructions_**

Olist voudrait se concentrer dans un premier temps sur le segment des **skis**. Les informations à étudier incluent : **le nombre d'annonces sur les skis**, les **prix** pratiqués, la **répartition de l’offre entre les différentes sous-catégories**, eventuellement la **profondeur des stocks** (= le nombre de produits disponibles, ou le nombre de tailles disponibles par produit), et tout ce que vous pensez qui serait **intéressant pour analyser le positionnement marché de FreeGlisse**.

Ces informations seraient d’autant plus intéressantes si elles étaient étudiées **par type de ski** (indiqué dans la Fiche Technique de chaque ski). N'hésitez pas à effectuer les regroupements qui vous semblent pertinents, à faire des analyses quantitatives et qualitatives, en utilisant **pandas** et en faisant de beaux **graphiques**. Pour les graphiques, vous utiliserez **Power BI**.

Côté technique, 2 compétences pourront être mises en oeuvre: le **crawling** entre les différentes pages du site, pour observer l’ensemble des offres de skis d’occasion (ici, pas besoin de selenium - des techniques plus simples seront possibles), et le **scraping des pages individuelles**, avec différentes infos à aller chercher.

Pour finir, l'objectif de ce cas est aussi de tirer de ces analyses des **insights business**, en particulier des **conclusions sur le positionnement commercial de FreeGlisse et les spécificités qu'il pourrait apporter à Olist**.
<br><br>

Pour extraire efficacement les informations des pages HTML, il est crucial de choisir les **bonnes méthodes de sélection** des éléments. Les méthodes **find()** et **find_all()** de BeautifulSoup sont couramment utilisées pour trouver des éléments dans un document HTML. Cependant, les méthodes **.select()** et **.select_one()** offrent une alternative plus **puissante et flexible**.

**_Présentation_**

En groupe, vous devrez **analyser les informations issues du scraping**, puis en fin de projet **présenter en environ 5 minutes** votre analyse à des Product Managers d'Olist (jurys de Databirdies), sous forme de **dashboard PowerBI**. L'essentiel est que la présentation soit **visuelle et intelligible par des stakeholders non-tech**.

### Utilité des Méthodes .select() et .select_one()

**.select():**

Cette méthode permet de sélectionner des éléments en utilisant des sélecteurs CSS, offrant ainsi une manière plus précise et flexible de cibler des éléments spécifiques dans le HTML

**.select_one():**

Cette méthode est similaire à .select(), mais elle ne retourne qu'un seul élément, le premier correspondant au sélecteur CSS fourni. C'est pratique lorsque vous savez qu'il n'y a qu'un seul élément correspondant au sélecteur ou lorsque vous souhaitez obtenir un élément spécifique sans avoir à parcourir une liste.

#### Avantages des Méthodes .select() et .select_one()

**_Flexibilité et Précision_**

Les sélecteurs CSS utilisés dans .select() permettent une sélection plus fine et plus flexible des éléments, surtout lorsque vous avez des structures HTML complexes.

**_Performances_**

Pour des opérations de sélection plus complexes ou spécifiques, .select() et .select_one() peuvent être plus performantes que find() et find_all(), car elles exploitent les sélecteurs CSS natifs qui sont optimisés pour ces tâches.

**_Syntaxe Moderne_**

L'utilisation des sélecteurs CSS est généralement plus intuitive et moderne, ce qui peut rendre le code plus lisible et facile à maintenir.

### C'est l'heure pour vous de scraper !

In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
import time
from datetime import datetime
from ast import literal_eval
from itertools import chain


In [ ]:
## Telechargement dataframe
df_ski_general = pd.read_excel('/content/df_general_ski.xlsx')
df_pratique = pd.read_excel('/content/df_pratique.xlsx')
df_qualité = pd.read_excel('/content/df_ski_qualite.xlsx')
df_qualité_2 = pd.read_excel('/content/df_ski_qualite.xlsx')
df_type = pd.read_excel('/content/df_type_de_ski.xlsx')

In [ ]:
## Supprimer colonne en trop
df_ski_general.drop(['Unnamed: 0'],axis = 1, inplace=True)
df_pratique.drop(['Index'],axis = 1, inplace=True)
df_qualité.drop(['Index'],axis = 1, inplace=True)
df_qualité_2.drop(['Index'],axis = 1, inplace=True)
df_type.drop(['Index'],axis = 1, inplace=True)

In [ ]:
## Supprimer doublon
df_ski_general = df_ski_general.drop_duplicates()
df_pratique = df_pratique.drop_duplicates()
df_qualité = df_qualité.drop_duplicates()
df_qualité_2 = df_qualité_2.drop_duplicates()
df_type = df_type.drop_duplicates()

In [ ]:
# JOINDRE LES TABLES SUR LA TABLE QUALITE
df = df_qualité.merge(df_pratique, on='ID', how='left')
df = df.merge(df_ski_general, on='ID', how='left')
df.drop(['url_y','Nom_y','url','Nom'],axis = 1, inplace=True)
df = df.merge(df_type, on='ID', how='left')
df.drop(['url','Nom'],axis = 1, inplace=True)
df.rename(columns={"url_x" : "url", 'Nom_x': 'Nom'}, inplace=True)
df_final = df


In [ ]:
## Remplir les vides par des "autres"
df_final ['Pratique']= df_final ['Pratique'].fillna("Autre")
df_final ['Categorie']= df_final ['Categorie'].fillna("Autre")
df_final

In [ ]:
## Informations sur les prix
df_final['Prix'].describe()

In [ ]:
# modifiaction du type de la clef primaire en STR
df_final['ID'] = df_final['ID'].astype(object)
df_final.info()

In [ ]:
pd.set_option('display.max_colwidth', None)

- Rossignol
- Salomon
- Vollkl
- Dynastar
- Fischer
- Elan
- Head

In [ ]:
## Liste des marques

Liste_brand = []
Compteur_Rossignol = 0
Compteur_Salomon = 0
Compteur_Volkl = 0
Compteur_Dynastar = 0
Compteur_Fischer = 0
Compteur_Elan = 0
Compteur_Head = 0
Compteur_Nordica= 0
Compteur_Kastle= 0
Compteur_Atomic= 0
Compteur_Blizzard= 0
Compteur_Tecnopro= 0
Compteur_K2= 0
Compteur_Exonde= 0
Compteur_Autre = 0
for x in df_final['Nom']:
  if re.search(r'\bRossignol\b', x):
    Liste_brand.append("Rossignol")
    print("Rossignol")
    Compteur_Rossignol += 1
  elif re.search(r'\bSalomon\b', x):
    Liste_brand.append("Salomon")
    print("Salomon")
    Compteur_Salomon += 1
  elif re.search(r'\bVolkl\b', x):
    Liste_brand.append("Volkl")
    print("Volkl")
    Compteur_Volkl += 1
  elif re.search(r'\bDynastar\b', x):
    Liste_brand.append("Dynastar")
    print("Dynastar")
    Compteur_Dynastar += 1
  elif re.search(r'\bFischer\b', x):
    Liste_brand.append("Fischer")
    print("Fischer")
    Compteur_Fischer += 1
  elif re.search(r'\bElan\b', x):
    Liste_brand.append("Elan")
    print("Elan")
    Compteur_Elan += 1
  elif re.search(r'\bHead\b', x):
    Liste_brand.append("Head")
    print("Head")
    Compteur_Head += 1
  elif re.search(r'\bNordica\b', x):
    Liste_brand.append("Nordica")
    print("Nordica")
    Compteur_Nordica += 1
  elif re.search(r'\bKastle\b', x):
    Liste_brand.append("Kastle")
    print("Kastle")
    Compteur_Kastle += 1
  elif re.search(r'\bAtomic\b', x):
    Liste_brand.append("Atomic")
    print("Atomic")
    Compteur_Atomic += 1
  elif re.search(r'\bBlizzard\b', x):
    Liste_brand.append("Blizzard")
    print("Blizzard")
    Compteur_Blizzard += 1
  elif re.search(r'\bTecnopro\b', x):
    Liste_brand.append("Tecnopro")
    print("Tecnopro")
    Compteur_Tecnopro += 1
  elif re.search(r'\bK2\b', x):
    Liste_brand.append("K2")
    print("K2")
    Compteur_K2 += 1
  elif re.search(r'\bExonde\b', x):
    Liste_brand.append("Exonde")
    print("Exonde")
    Compteur_Exonde += 1
  else:
    Liste_brand.append("Autre")
    print("Autre")
    Compteur_Autre += 1

print(" ")
print("Rossignol :", Compteur_Rossignol)
print("Salomon :", Compteur_Salomon)
print("Volkl :", Compteur_Volkl)
print("Dynastar :", Compteur_Dynastar)
print("Fischer :", Compteur_Fischer)
print("Elan :", Compteur_Elan)
print("Head :", Compteur_Head)
print("Nordica :", Compteur_Nordica)
print("Kastle :", Compteur_Kastle)
print("Atomic :", Compteur_Atomic)
print("Blizzard :", Compteur_Blizzard)
print("Tecnopro :", Compteur_Tecnopro)
print("K2 :", Compteur_K2)
print("Exonde :", Compteur_Exonde)
print("Autre :", Compteur_Autre)
print(Liste_brand)
df_final["marque"] = Liste_brand

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
df_final["Nom_X"] = df_final["Nom"]
test = df_final

In [ ]:
# Mettre à jour dans une nouvelle colonne : Nom ski nettoyé
test["Nom_X"] = test["Nom_X"].str.replace(r'\s*\+.*$', '', regex=True)
test["Nom_X"] = test["Nom_X"].str.replace(r'Ski occasion ', '', regex=True)
test["Nom_X"] = test["Nom_X"].str.replace(r'junior', '', regex=True, flags = re.IGNORECASE)
test["Nom_X"] = test["Nom_X"].str.replace(r'\.\.\.', '', regex=True)
df_final["Nom_X"] = test["Nom_X"]

In [ ]:
df_final = df_final.rename(columns={"Nom": "Nom_Masquer", "Nom_X": "Nom"})
df_final.head()

In [ ]:
df_final.describe()

In [ ]:
# boite a moustache
df_final.boxplot(column=['Prix'])

In [ ]:
# Calculer le premier et le troisième quartile (Q1 et Q3)
Q1 = df_final['Prix'].quantile(0.25)
Q3 = df_final['Prix'].quantile(0.75)

# Calculer l'IQR (écart interquartile)
IQR = Q3 - Q1

# Définir les bornes pour les valeurs aberrantes
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identifier les valeurs aberrantes
outliers = df_final[(df_final['Prix'] < lower_bound) |
                                 (df_final['Prix'] > upper_bound)]

# Afficher le nombre de valeurs aberrantes et les valeurs elles-mêmes
print("Nombre de valeurs aberrantes dans 'Prix':", len(outliers))


In [ ]:
upper_bound = Q3 + 1.5 * IQR
upper_bound

In [ ]:
# visualisation des produits par qualite
df_final.groupby('Qualite').size().plot(kind='bar')

In [ ]:
df_final.boxplot(column=['Prix'], by='Qualite')

In [ ]:
# Effectuer une correlation entre qualite et pratique
df_final.groupby(['Qualite', 'Pratique']).size().unstack().plot(kind='bar', stacked=True)

In [ ]:
# Effectuer une correlation entre qualite et pratrique => voir cour encoding netflix type abonnement



In [ ]:
df_final.to_csv("df_final.csv", encoding = "utf-8")